# Market Data Loading

Pull market data from S3 via the Java MarketDataEntry infrastructure using JPype.

**Prerequisites:**
- `gnome-backtest` uber JAR built: `cd gnome-backtest && mvn package -DskipTests`
- AWS credentials configured (`~/.aws/credentials` or env vars)

In [21]:
%load_ext autoreload
%autoreload 2

from datetime import datetime

from gnomepy.java import (
    ensure_jvm_started,
    MarketDataClient,
    SchemaType,
)

ensure_jvm_started()

Failed to reload module 'gnomepy.java.market_data' from file '/Users/mprey/Programming/gnome-trading-group/gnomepy/gnomepy/java/market_data.py'
Traceback (most recent call last):
  File "/Users/mprey/Library/Caches/pypoetry/virtualenvs/gnomepy-vcmygBcy-py3.13/lib/python3.13/site-packages/IPython/extensions/autoreload.py", line 325, in check
    superreload(m, reload, self.old_objects)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/mprey/Library/Caches/pypoetry/virtualenvs/gnomepy-vcmygBcy-py3.13/lib/python3.13/site-packages/IPython/extensions/autoreload.py", line 584, in superreload
    module = reload(module)
  File "/opt/homebrew/Cellar/python@3.13/3.13.11/Frameworks/Python.framework/Versions/3.13/lib/python3.13/importlib/__init__.py", line 129, in reload
    _bootstrap._exec(spec, module)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 866, in _exec
  File "<frozen importlib._bootstrap_external>", line 1023, in exec_module
  File "<frozen

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


[autoreload of gnomepy failed: Traceback (most recent call last):
  File "/Users/mprey/Library/Caches/pypoetry/virtualenvs/gnomepy-vcmygBcy-py3.13/lib/python3.13/site-packages/IPython/extensions/autoreload.py", line 325, in check
    superreload(m, reload, self.old_objects)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/mprey/Library/Caches/pypoetry/virtualenvs/gnomepy-vcmygBcy-py3.13/lib/python3.13/site-packages/IPython/extensions/autoreload.py", line 584, in superreload
    module = reload(module)
  File "/opt/homebrew/Cellar/python@3.13/3.13.11/Frameworks/Python.framework/Versions/3.13/lib/python3.13/importlib/__init__.py", line 129, in reload
    _bootstrap._exec(spec, module)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 866, in _exec
  File "<frozen importlib._bootstrap_external>", line 1023, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/Users/mprey/Programming/gnome-trading-gr

## Configuration

Fill in your exchange, security, schema type, and date range.

In [ ]:
# --- Fill these in ---
EXCHANGE_ID = 1
SECURITY_ID = 2
SCHEMA_TYPE = SchemaType.MBP_10
START = datetime(2026, 1, 20, 10, 30)
END = datetime(2026, 1, 20, 12, 0)
BUCKET = "gnome-market-data-prod"
# ---------------------

## Load as Python Schema Objects

In [ ]:
client = MarketDataClient(bucket=BUCKET)
schemas = client.load(
    exchange_id=EXCHANGE_ID,
    security_id=SECURITY_ID,
    schema_type=SCHEMA_TYPE,
    start=START,
    end=END,
)
print(f"Loaded {len(schemas)} records")

In [ ]:
# Inspect the first record
if schemas:
    s = schemas[0]
    print(f"Schema type: {s.schema_type}")
    print(f"Event timestamp: {s.event_timestamp}")
    print(f"Full dict: {s.to_dict()}")

## Load as DataFrame

In [ ]:
df = client.load_dataframe(
    exchange_id=EXCHANGE_ID,
    security_id=SECURITY_ID,
    schema_type=SCHEMA_TYPE,
    start=START,
    end=END,
)
print(f"DataFrame shape: {df.shape}")
df.head()

## Available Schema Types

```
SchemaType.MBO       - Market by Order
SchemaType.MBP_10    - Market by Price (10 levels)
SchemaType.MBP_1     - Market by Price (1 level)
SchemaType.BBO_1S    - Best Bid/Offer (1s)
SchemaType.BBO_1M    - Best Bid/Offer (1m)
SchemaType.TRADES    - Trade messages
SchemaType.OHLCV_1S  - OHLCV (1s bars)
SchemaType.OHLCV_1M  - OHLCV (1m bars)
SchemaType.OHLCV_1H  - OHLCV (1h bars)
```

## Load Multiple Schema Types

In [ ]:
# Example: load trades alongside MBP data
trades_df = client.load_dataframe(
    exchange_id=EXCHANGE_ID,
    security_id=SECURITY_ID,
    schema_type=SchemaType.TRADES,
    start=START,
    end=END,
)
print(f"Trades: {len(trades_df)} records")
trades_df.head()

## Price Scaling

Raw prices are scaled integers (x 1,000,000,000). Divide to get human-readable values.

In [ ]:
PRICE_SCALE = 1_000_000_000

if not df.empty and "bid_price_0" in df.columns:
    df["mid_price"] = (df["bid_price_0"] + df["ask_price_0"]) / (2 * PRICE_SCALE)
    df["spread"] = (df["ask_price_0"] - df["bid_price_0"]) / PRICE_SCALE
    print(df[["mid_price", "spread"]].describe())

## Quick Plot

In [ ]:
import matplotlib.pyplot as plt

if not df.empty and "mid_price" in df.columns:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    ax1.plot(df["mid_price"], linewidth=0.5)
    ax1.set_ylabel("Mid Price")
    ax1.set_title(f"Exchange {EXCHANGE_ID} / Security {SECURITY_ID}")

    ax2.plot(df["spread"], linewidth=0.5, color="orange")
    ax2.set_ylabel("Spread")
    ax2.set_xlabel("Record Index")

    plt.tight_layout()
    plt.show()

In [ ]:
  import requests

  # Hyperliquid info endpoint
  resp = requests.post("https://api.hyperliquid.xyz/info", json={"type": "meta"})
  meta = resp.json()

  for asset in meta["universe"]:
      if asset["name"] in ["ETH", "BTC"]:
          sz_decimals = asset["szDecimals"]
          max_decimals = 6  # perps
          price_decimals = max_decimals - sz_decimals
          tick_size = 10 ** (-price_decimals)
          lot_size = 10 ** (-sz_decimals)
          print(f"{asset['name']}: tick=${tick_size}, lot={lot_size}, szDecimals={sz_decimals}")